In [ ]:
# Install required libraries
!pip install -q timm pillow-avif-plugin

import os, shutil, zipfile, random, warnings
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pillow_avif  # Adds AVIF support to PIL
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from google.colab import files

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Upload your Road_Verification.zip file ──
print('📁 Please upload your Road_Verification.zip file...')
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
print(f'✅ Uploaded: {zip_name}')

In [ ]:
# ── Extract ZIP and organize dataset ──
EXTRACT_DIR = '/content/Road_Verification'
DATA_DIR    = '/content/dataset'
TRAIN_DIR   = os.path.join(DATA_DIR, 'train')
VAL_DIR     = os.path.join(DATA_DIR, 'val')
CLASSES     = ['Road_block', 'Road_clear']
SPLIT_RATIO = 0.8   # 80% train, 20% val
SEED        = 42
random.seed(SEED)

# Extract
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')
print('✅ ZIP extracted')

# Handle nested folder (some zips have an extra root folder)
if not os.path.exists(os.path.join(EXTRACT_DIR, 'Road_block')):
    # Find the actual folder
    for root, dirs, _ in os.walk('/content/'):
        if 'Road_block' in dirs and 'Road_clear' in dirs:
            EXTRACT_DIR = root
            break

print(f'📂 Dataset found at: {EXTRACT_DIR}')

# Supported image extensions
VALID_EXT = {'.jpg', '.jpeg', '.png', '.webp', '.avif', '.bmp'}

def get_images(folder):
    imgs = []
    for f in os.listdir(folder):
        if os.path.splitext(f)[1].lower() in VALID_EXT:
            imgs.append(os.path.join(folder, f))
    return imgs

# Build train/val split
for split in ['train', 'val']:
    for cls in CLASSES:
        os.makedirs(os.path.join(DATA_DIR, split, cls), exist_ok=True)

total = {}
for cls in CLASSES:
    src_folder = os.path.join(EXTRACT_DIR, cls)
    imgs = get_images(src_folder)
    random.shuffle(imgs)
    split_idx = int(len(imgs) * SPLIT_RATIO)
    train_imgs, val_imgs = imgs[:split_idx], imgs[split_idx:]

    for src in train_imgs:
        shutil.copy(src, os.path.join(TRAIN_DIR, cls, os.path.basename(src)))
    for src in val_imgs:
        shutil.copy(src, os.path.join(VAL_DIR, cls, os.path.basename(src)))

    total[cls] = {'train': len(train_imgs), 'val': len(val_imgs)}
    print(f'  {cls}: {len(train_imgs)} train | {len(val_imgs)} val')

print(f'\n✅ Dataset organized into train/val splits')

In [ ]:
# EfficientNet-B3 expects 300x300 input
IMG_SIZE   = 300
BATCH_SIZE = 16

# ── Custom Dataset (handles .webp, .avif etc.) ──
class RoadDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.class_to_idx = {cls: i for i, cls in enumerate(sorted(CLASSES))}
        for cls in CLASSES:
            folder = os.path.join(root_dir, cls)
            label  = self.class_to_idx[cls]
            for fname in os.listdir(folder):
                if os.path.splitext(fname)[1].lower() in VALID_EXT:
                    self.samples.append((os.path.join(folder, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception:
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE), (128, 128, 128))
        if self.transform:
            img = self.transform(img)
        return img, label

# ── Transforms ──
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 20, IMG_SIZE + 20)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

train_dataset = RoadDataset(TRAIN_DIR, transform=train_transform)
val_dataset   = RoadDataset(VAL_DIR,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

CLASS_NAMES = sorted(CLASSES)  # ['Road_block', 'Road_clear']
print(f'Class mapping: {dict(enumerate(CLASS_NAMES))}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

# ── Visualize sample images ──
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for cls_idx, cls in enumerate(CLASS_NAMES):
    folder = os.path.join(TRAIN_DIR, cls)
    imgs = [f for f in os.listdir(folder) if os.path.splitext(f)[1].lower() in VALID_EXT][:4]
    for i, fname in enumerate(imgs):
        try:
            img = Image.open(os.path.join(folder, fname)).convert('RGB')
            axes[cls_idx][i].imshow(img)
            axes[cls_idx][i].set_title(cls.replace('_', ' '), fontsize=10,
                                       color='red' if 'block' in cls else 'green')
            axes[cls_idx][i].axis('off')
        except:
            pass
plt.suptitle('Sample Training Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Build Model ──
def build_model(num_classes=2):
    model = timm.create_model(
        'efficientnet_b3',
        pretrained=True,
        num_classes=num_classes
    )
    return model

model = build_model().to(device)

# ── Count parameters ──
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ EfficientNet-B3 loaded (pretrained on ImageNet)')
print(f'   Total params:     {total_params:,}')
print(f'   Trainable params: {trainable_params:,}')
print(f'   Classes: {CLASS_NAMES}')

In [ ]:
# ── Training Configuration ──
EPOCHS    = 20
LR        = 3e-4
PATIENCE  = 5    # early stopping

# Loss: use label smoothing to reduce overconfidence on small dataset
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer: AdamW with weight decay
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# LR Scheduler: cosine annealing
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# ── Training & Evaluation Functions ──
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        loss    = criterion(outputs, labels)

        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += imgs.size(0)
    return total_loss / total, correct / total

print('✅ Training config ready!')
print(f'   Epochs: {EPOCHS} | LR: {LR} | Batch: {BATCH_SIZE}')

In [ ]:
# ── Training Loop with Early Stopping ──
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0
MODEL_SAVE_PATH = '/content/road_classifier_efficientnet_b3.pth'

print('🚀 Starting Training...\n')
print(f'{"Epoch":>6} | {"Train Loss":>10} | {"Train Acc":>9} | {"Val Loss":>8} | {"Val Acc":>7} | {"LR":>10}')
print('-' * 65)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss,   val_acc   = eval_epoch(model, val_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    current_lr = optimizer.param_groups[0]['lr']
    flag = ''

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'class_names': CLASS_NAMES
        }, MODEL_SAVE_PATH)
        flag = '  ✅ BEST'
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\n⏹️  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

    print(f'{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2%} | {val_loss:>8.4f} | {val_acc:>6.2%} | {current_lr:>10.2e}{flag}')

print(f'\n🏆 Best Validation Accuracy: {best_val_acc:.2%}')
print(f'💾 Best model saved to: {MODEL_SAVE_PATH}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_ran, history['train_loss'], 'b-o', label='Train Loss', markersize=4)
ax1.plot(epochs_ran, history['val_loss'],   'r-o', label='Val Loss',   markersize=4)
ax1.set_title('Loss Curve', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_ran, [a*100 for a in history['train_acc']], 'b-o', label='Train Acc', markersize=4)
ax2.plot(epochs_ran, [a*100 for a in history['val_acc']],   'r-o', label='Val Acc',   markersize=4)
ax2.axhline(best_val_acc*100, color='green', linestyle='--', alpha=0.5, label=f'Best: {best_val_acc:.1%}')
ax2.set_title('Accuracy Curve', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('EfficientNet-B3 Training — Road Block Classifier', fontsize=14)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Training curves saved.')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best model
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print('\n📊 Classification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Inference Function ──
def predict_image(image_path, model, class_names, threshold=0.5):
    """
    Predict whether a road image is blocked or clear.
    Returns: predicted class, confidence, and all class probabilities.
    """
    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])

    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)  # (1, C, H, W)

    model.eval()
    with torch.no_grad():
        output = model(tensor)
        probs  = torch.softmax(output, dim=1).cpu().squeeze().numpy()

    pred_idx  = probs.argmax()
    pred_class = class_names[pred_idx]
    confidence = probs[pred_idx]

    return pred_class, confidence, probs, img


def show_prediction(image_path, model, class_names):
    pred_class, confidence, probs, img = predict_image(image_path, model, class_names)

    is_blocked = 'block' in pred_class.lower()
    color      = 'red' if is_blocked else 'green'
    icon       = '🚧' if is_blocked else '✅'
    label      = 'ROAD BLOCKED' if is_blocked else 'ROAD CLEAR'

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5),
                                    gridspec_kw={'width_ratios': [2, 1]})

    # Show image
    ax1.imshow(img)
    ax1.set_title(f'{icon}  {label}\nConfidence: {confidence:.1%}',
                  fontsize=15, fontweight='bold', color=color)
    ax1.axis('off')

    # Probability bar chart
    bars = ax2.barh(class_names, probs * 100,
                    color=['#e74c3c' if 'block' in c.lower() else '#2ecc71'
                           for c in class_names])
    ax2.set_xlim(0, 100)
    ax2.set_xlabel('Probability (%)', fontsize=11)
    ax2.set_title('Class Probabilities', fontsize=13, fontweight='bold')
    for bar, prob in zip(bars, probs):
        ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{prob:.1%}', va='center', fontsize=12, fontweight='bold')
    ax2.grid(axis='x', alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f'\n🔍 Prediction: {icon} {label}')
    print(f'   Confidence: {confidence:.2%}')
    for cls, prob in zip(class_names, probs):
        bar = '█' * int(prob * 30)
        print(f'   {cls:15s}: {bar} {prob:.2%}')

print('✅ Inference functions ready!')

In [ ]:
# ── Upload & Classify Your Image ──
# Load best model first
checkpoint = torch.load(MODEL_SAVE_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
CLASS_NAMES = checkpoint.get('class_names', CLASS_NAMES)

print('📸 Upload a road image to classify...')
uploaded_img = files.upload()

for fname in uploaded_img.keys():
    print(f'\n🔎 Classifying: {fname}')
    show_prediction(fname, model, CLASS_NAMES)

In [ ]:
# ── Batch Inference: Classify Multiple Images ──
# Run this cell to upload and classify multiple images at once

print('📸 Upload multiple road images for batch classification...')
uploaded_imgs = files.upload()

results = []
for fname in uploaded_imgs.keys():
    pred_class, confidence, probs, _ = predict_image(fname, model, CLASS_NAMES)
    is_blocked = 'block' in pred_class.lower()
    results.append({
        'File': fname,
        'Prediction': '🚧 BLOCKED' if is_blocked else '✅ CLEAR',
        'Confidence': f'{confidence:.2%}'
    })

print('\n📋 Batch Results:')
print(f'{"File":<40} {"Prediction":<15} {"Confidence"}')
print('-' * 70)
for r in results:
    print(f'{r["File"]:<40} {r["Prediction"]:<15} {r["Confidence"]}')

In [ ]:
# Download the best model checkpoint
print(f'📥 Downloading model: {MODEL_SAVE_PATH}')
files.download(MODEL_SAVE_PATH)
print('✅ Model downloaded!')

# Also download plots
files.download('/content/training_curves.png')
files.download('/content/confusion_matrix.png')

In [ ]:
# ── Load from checkpoint (skip training) ──
# 1. Upload your saved .pth file
print('📂 Upload your saved model .pth file...')
uploaded_model = files.upload()
pth_file = list(uploaded_model.keys())[0]

# 2. Rebuild model and load weights
model_loaded = build_model(num_classes=2).to(device)
ckpt = torch.load(pth_file, map_location=device)
model_loaded.load_state_dict(ckpt['model_state_dict'])
LOADED_CLASS_NAMES = ckpt.get('class_names', ['Road_block', 'Road_clear'])
print(f'✅ Model loaded! Classes: {LOADED_CLASS_NAMES}')
print(f'   Best val accuracy: {ckpt.get("val_acc", "N/A")}')

# 3. Upload image and predict
print('\n📸 Upload an image to classify...')
test_imgs = files.upload()
for fname in test_imgs.keys():
    show_prediction(fname, model_loaded, LOADED_CLASS_NAMES)